# Project 1: CNN from Scratch (Baseline)
This notebook fulfills Requirement #5: Train a CNN-based architecture from scratch and compare with the pre-trained model.

In [5]:
!pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.0 MB 904.2 kB/s eta 0:00:09
   --- ------------------------------------ 0.8/8.0 MB 1.0 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.0 MB 1.0 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.0 MB 1.0 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.0 MB 1.0 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.0 MB 1.0 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/8.0 MB 835.6 kB/s eta 0:00:08
   --------- ------------------------------ 1.8/8.0 MB 809.6 kB/s eta 0:00:08
   --------- ----------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


### 1. Load and Preprocess CIFAR-10
Note: Since we are building from scratch, we do not need to resize the images to 299x299. We will use the native 32x32 size to train faster.

In [6]:
cifar10 = tf.keras.datasets.cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Split training into train and validation (80/20)
from sklearn.model_selection import train_test_split
x_train_split, x_val_split, y_train_split, y_val_split = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

print(f"Training set: {x_train_split.shape}")
print(f"Validation set: {x_val_split.shape}")
print(f"Test set: {x_test.shape}")

Training set: (40000, 32, 32, 3)
Validation set: (10000, 32, 32, 3)
Test set: (10000, 32, 32, 3)


### 2. Define the CNN Architecture (From Scratch)

In [7]:
# Define data augmentation
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal", seed=42),
  tf.keras.layers.RandomRotation(0.05, seed=42),
  tf.keras.layers.RandomZoom(0.1, seed=42),
])

# Build the CNN using the Sequential API
model = tf.keras.Sequential([
    # Input layer for 32x32 RGB images
    tf.keras.Input(shape=(32, 32, 3)),
    
    # Apply augmentation directly inside the model
    data_augmentation,
    
    # Block 1
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # Block 2
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # Block 3
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # Classifier Head
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5), # Helps prevent overfitting
    tf.keras.layers.Dense(10, activation="softmax") # 10 classes for CIFAR-10
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 356,810 (1.36 MB)

 Trainable params: 356,810 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

### 3. Compile and Train the Model
Since we are training from scratch, we can use the Adam optimizer which generally converges faster than SGD for un-initialized weights.

In [8]:
# Compile the model
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Add Early Stopping to prevent useless training if it plateaus
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    x_train_split, y_train_split, 
    epochs=30, 
    validation_data=(x_val_split, y_val_split),
    callbacks=[early_stopping]
)

Epoch 1/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 48s 35ms/step - accuracy: 0.3738 - loss: 1.7176 - val_accuracy: 0.5243 - val_loss: 1.3055
Epoch 2/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 47s 38ms/step - accuracy: 0.5067 - loss: 1.3846 - val_accuracy: 0.5832 - val_loss: 1.1730
Epoch 3/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 48s 39ms/step - accuracy: 0.5593 - loss: 1.2434 - val_accuracy: 0.5876 - val_loss: 1.2263
Epoch 4/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 47s 38ms/step - accuracy: 0.5970 - loss: 1.1446 - val_accuracy: 0.6560 - val_loss: 0.9789
Epoch 5/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 47s 38ms/step - accuracy: 0.6232 - loss: 1.0814 - val_accuracy: 0.6709 - val_loss: 0.9447
Epoch 6/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 49s 39ms/step - accuracy: 0.6375 - loss: 1.0319 - val_accuracy: 0.6926 - val_loss: 0.8835
Epoch 7/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 48s 38ms/step - accuracy: 0.6541 - loss: 0.9875 - val_accuracy: 0.7104 - val_loss: 0.8460
Epoch 8/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 48s 38ms/step - accuracy: 0.6671 -

In [9]:
# Evaluate on the Test Set
test_loss, test_acc = model.evaluate(x_test,  y_test, verbose=2)
print(f"\nTest Accuracy (CNN from Scratch): {test_acc*100:.2f}%")

313/313 - 3s - 9ms/step - accuracy: 0.7403 - loss: 0.7678

Test Accuracy (CNN from Scratch): 74.03%
